# Hangman: neural sequence modeling

This notebook runs the conditioned character Transformer pipeline using the
same CLI scripts as the local workflow. The current recipe uses per-position
supervision with policy and presence losses disabled. The optional offline
teacher supplies rollout states; additional states are generated without it.
Inference uses only the network, with no dictionary or oracle.

Run with FAST = True first to exercise the pipeline, then set it to False for
the full 512-dimensional, 40,000-step recipe. Use SPLIT_SEED = 7 and TRAIN_SEED = 1.
See README.md and reports/kaggle_run.md for the included result evidence.


In [ ]:
# ── Setup: locate inputs, put the package on the path ───────────────────────────
import glob, os, subprocess, sys, time

FAST = True          # ← flip to False for the full run
TRAIN_HERE = True    # ← False to skip training and use an uploaded checkpoint

# Which attached .pt to use when TRAIN_HERE=False. Substring match on the file
# name, e.g. "wide-512". Leave "" to take the first checkpoint found, which is
# only safe when exactly one is attached.
CHECKPOINT = ""

# TWO DIFFERENT SEEDS — conflating them silently corrupts the held-out metric.
# SPLIT_SEED partitions train/val and MUST match build_labels.py and
# evaluate.py, which both default to 7. Passing the training seed here instead
# repartitions the data, so the model trains on words evaluate.py scores it on.
SPLIT_SEED = 7
TRAIN_SEED = 1       # weight init and batch order only

# Recorded 40,000-step recipe; see reports/kaggle_run.md for validation metrics.
# Keep the settings together so local and notebook workflows agree.
STATES_PER_WORD = 25                       # -> ~5.5M states + labels carried
ARCH   = ["--dim", 512, "--ff", 2048]      # approximately 19.01M parameters
LOSS   = ["--w-perpos", 1.0, "--w-policy", 0, "--w-presence", 0]
STEPS  = ["--steps", 40000, "--warmup", 1000]
BATCH  = ["--batch-size", 256]

def _tree(root="/kaggle/input", depth=5, max_files=8):
    """Print what is actually mounted — the fastest way to resolve a bad path."""
    if not os.path.isdir(root):
        print(f"  {root} does not exist — nothing is attached.")
        return
    for cur, dirs, files in os.walk(root):
        level = cur[len(root):].count(os.sep)
        pad = "  " * level
        print(f"{pad}{os.path.basename(cur) or root}/")
        for f in sorted(files)[:max_files]:
            print(f"{pad}  {f}")
        if len(files) > max_files:
            print(f"{pad}  ... {len(files) - max_files} more files")
        if level + 1 >= depth:
            if dirs:
                print(f"{pad}  ... {len(dirs)} more dirs (depth limit)")
            dirs[:] = []

# Find the package by a file we know it contains, at ANY nesting depth. Kaggle
# may or may not preserve the archive's folder structure, so do not assume one.
_hits = sorted(glob.glob("/kaggle/input/**/hangman/tokenizer.py", recursive=True))
if not _hits:
    print("Could not find the source package. Mounted under /kaggle/input:\n")
    _tree()
    raise SystemExit(
        "\nAttach the 'hangman-src' dataset (Add Input). If it IS listed above, "
        "restart the session — inputs attached to a running kernel are not "
        "always mounted until it restarts."
    )

SRC = os.path.dirname(os.path.dirname(_hits[0]))   # .../src
PKG = os.path.dirname(SRC)                         # dataset root
sys.path.insert(0, SRC)

# The competition data (train.txt / test.txt live in the competition input)
def _find(name):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)
            if not p.startswith(PKG)]
    return sorted(hits)[0] if hits else None

TRAIN, TEST = _find("train.txt"), _find("test.txt")
if not TRAIN:
    print("train.txt not found. Mounted under /kaggle/input:\n")
    _tree()
    raise SystemExit(
        "\nAttach the COMPETITION data: Add Input -> the 'Competitions' tab "
        "(not 'Datasets') -> select this hackathon. Your own uploaded dataset "
        "does not include train.txt/test.txt by design — competition data is "
        "never redistributed in our archive.")

WORK = "/kaggle/working"
MODEL = f"{WORK}/model.pt"
os.makedirs(f"{WORK}/experiments", exist_ok=True)
os.chdir(WORK)

# If not training here, find a checkpoint in the attached inputs.
UPLOADED = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
if not TRAIN_HERE:
    if not UPLOADED:
        print("No .pt found. Mounted under /kaggle/input:\n"); _tree()
        raise SystemExit(
            "\nTRAIN_HERE=False but no checkpoint is attached. "
            "Add Input -> Your Work -> Notebook Output.")
    matches = [p for p in UPLOADED
               if CHECKPOINT in os.path.basename(p)] if CHECKPOINT else UPLOADED
    if not matches:
        raise SystemExit(
            f"No attached checkpoint matches CHECKPOINT={CHECKPOINT!r}.\n"
            + "\n".join(f"  {p}" for p in UPLOADED))
    if len(matches) > 1:
        print(f"WARNING: {len(matches)} checkpoints match; using the first. "
              f"Set CHECKPOINT to disambiguate:")
        for p in matches:
            print(f"    {os.path.basename(p)}")
    MODEL = matches[0]

import torch
print(f"package : {PKG}")
print(f"src     : {SRC}")
print(f"train   : {TRAIN}")
print(f"test    : {TEST}")
print(f"model   : {MODEL}{'' if TRAIN_HERE else '  (uploaded)'}")
print(f"device  : {'cuda' if torch.cuda.is_available() else 'cpu'} "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — fine'})")
print(f"cores   : {os.cpu_count()}   |   FAST={FAST}   TRAIN_HERE={TRAIN_HERE}")

# Fail fast if the package is present but broken, rather than three cells later.
from hangman.dataset import load_words          # noqa: E402
print("package import OK")

def run(script, *args):
    """Run a pipeline stage, streaming its output."""
    cmd = [sys.executable, f"{PKG}/scripts/{script}", *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    t = time.time()
    r = subprocess.run(cmd, text=True)
    print(f"[{script} finished in {time.time()-t:.0f}s]", flush=True)
    if r.returncode:
        raise RuntimeError(f"{script} failed with code {r.returncode}")

In [ ]:
# ── Sanity check the data before spending hours on it ──────────────────────────
from hangman.dataset import load_words, split_words, describe

train_words = load_words(TRAIN)
print("train:", describe(train_words))

if TEST:
    test_words = load_words(TEST)
    print("test :", describe(test_words))
    overlap = len(set(train_words) & set(test_words))
    print(f"\ntrain/test overlap: {overlap} words "
          f"({overlap/max(len(test_words),1):.2%})")
    print("Near-zero overlap is expected and is what the leave-one-out teacher "
          "is designed for: the student never relies on having seen the word.")

# The validation split is held out of BOTH the student's states and the
# teacher's vocabulary — the only honest proxy for the private set.
tr, va = split_words(train_words, val_size=5000, seed=7)
print(f"\ntrain split {len(tr)} | val split {len(va)} (held out everywhere)")

In [ ]:
# ── Stage 1: training states ──────────────────────────────────────────────────
# The shipped recipe is --w-policy 0 --w-presence 0, so the ONLY live target is
# per-position: the true letter at each hidden position, read straight off the
# word by encode_targets. No oracle is involved.
#
# That makes the leave-one-out teacher optional. It is kept because its
# on-policy rollouts are a genuinely different state distribution, but the bulk
# of the data now comes from build_states.py, which needs no teacher and runs in
# ~90 seconds instead of ~22 minutes.
#
# Total rows include carried rollout states and can overlap generated states.
if TRAIN_HERE:
    run("build_labels.py",
        "--train", TRAIN,
        "--out", "experiments/labels.tsv",
        "--workers", max(os.cpu_count() - 1, 1),
        "--seed", SPLIT_SEED,
        *(["--fast"] if FAST else ["--full"]))
    run("build_states.py",
        "--train", TRAIN,
        "--out", f"{WORK}/states.tsv",
        "--per-word", 4 if FAST else STATES_PER_WORD,
        "--seed", SPLIT_SEED,
        "--include", "experiments/labels.tsv")
else:
    print("TRAIN_HERE=False — using the uploaded checkpoint, skipping states.")


In [ ]:
# ── Stage 2: train the student ────────────────────────────────────────────────
# Ends with a hard golden-word gate: eight single-letter probes, scored by mean
# rank. It raises rather than warns — a silent index-space bug once trained a
# garbage model through a full run while reporting healthy loss curves.
#
# ~2.4 h on a T4. FAST shrinks everything to a few minutes for a smoke test and
# does NOT reproduce the submitted model.
if TRAIN_HERE:
    run("train.py",
        "--labels", f"{WORK}/states.tsv",
        "--out", MODEL,
        "--seed", TRAIN_SEED,
        *BATCH, *LOSS,
        *(["--fast", "--skip-golden"] if FAST else ARCH + STEPS))
else:
    print(f"TRAIN_HERE=False — skipping training, using {MODEL}")


In [ ]:
# ── Stage 3: score on the held-out split, with confidence intervals ───────────
# Never promote a change whose gain sits inside the CI. Record the seed spread:
# the organizers re-train this pipeline, and an unstated variance reads as a
# discrepancy rather than as noise.
run("evaluate.py",
    "--model", MODEL,
    "--train", TRAIN,
    "--seed", SPLIT_SEED,
    "--n", 500 if FAST else 2000,
    "--ledger", "experiments/results.jsonl",
    *([] if FAST else ["--seeds", 1, 2, 3]))

In [ ]:
# ── Stage 4: play every test word and write submission.csv ────────────────────
# Batched lockstep play: one forward pass per TURN across the whole batch, so
# 250k games cost ~max-guesses passes rather than ~2.5M single-sequence ones.
if TEST:
    run("make_submission.py",
        "--model", MODEL,
        "--test", TEST,
        "--out", f"{WORK}/submission.csv")
else:
    print("No test.txt attached — skipping submission.")

In [ ]:
# ── Final check: the file that actually gets scored ───────────────────────────
import csv
if TEST:
    from hangman.submit import validate_submission
    n = sum(1 for _ in open(TEST) if _.strip())
    print(validate_submission(f"{WORK}/submission.csv", n))
    with open(f"{WORK}/submission.csv") as fh:
        for i, row in zip(range(4), csv.reader(fh)):
            print(row)
    print("...")
    print("\nReminder: the self-play rate printed above is measured on the PUBLIC "
          "test words and is not an estimate of private-set performance. "
          "The held-out number from Stage 3 is the one to trust.")